# Bias correct and downscaling of PM2.5

This step includes downscaling the climate model data to the same grid as the observations (0.1°x0.1°)

In [ ]:
import os
import xarray as xr
import warnings
from utils.utils import adjust_longitude, bilinear_interp, get_scenario_config, standardise_latlon
import config
from utils.utils import require_dir
import pathlib

In [ ]:
warnings.filterwarnings('ignore')

# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245"

configs = get_scenario_config(model, scenario)
ensemble_members = configs["ensemble_members"]
years = configs["years"]

PM25_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / model / "pm25" / "annual_pm25")
OBS_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "PM2.5_obs")
SAVE_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / model / "pm25" / "annual_pm25_bc")

# === Main loop ===
for ens_num in ensemble_members:
    print(f"Processing {scenario}, Ensemble {ens_num:02d}")
    # Load data arrays
    dates = f"{years.start}-{years.stop}"

    pm25_file = f"Annual_PM25_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    pm25_path = os.path.join(PM25_DIR, pm25_file)
    pm25 = xr.open_dataarray(pm25_path)

    hist_file = "Annual_PM25_CESM2_hist_01_1990-2010.nc"
    hist_path = os.path.join(PM25_DIR, hist_file)
    hist = xr.open_dataarray(hist_path)

    obs_file = "DIMAQ_PM25_1990-2016.nc"
    obs_path = os.path.join(OBS_DIR, obs_file)
    obs = xr.open_dataset(obs_path)["Mean"]

    # Baseline years for fi_2000 and historical
    base = slice("1990", "2010")
    hist_base = hist.sel(year=base).mean("year")
    obs_base = obs.sel(year=base).mean("year")
    # Change lat/lon coordinate names to lat, lon
    obs_base = standardise_latlon(obs_base)

    # Calculate delta
    delta_fi = adjust_longitude(pm25 / hist_base)

    # Interpolate to the new grid
    regridder = bilinear_interp(delta_fi, obs_base)
    ds_delta_fi = regridder(delta_fi)

    # Bias correct delta
    bc_pm25 = obs_base * ds_delta_fi

    out_file = f"Annual_PM25_BC_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    out_path = os.path.join(SAVE_DIR, out_file)

    print(f"Saving to {out_path}")
    description = ("Annual mean PM2.5 bias corrected to DIMAQ data: "
                   "Gavin Shaddick, Matthew L. Thomas, Heresh Amini, David "
                   "Broday, Aaron Cohen, Joseph Frostad, Amelia Green, Sophie "
                   "Gumy, Yang Liu, Randall V. Martin, Annette Pruss-Ustun, "
                   "Daniel Simpson, Aaron van Donkelaar, and Michael Brauer "
                   "Environmental Science & Technology 2018 52 (16), 9069-9078"
                   "DOI: 10.1021/acs.est.8b02864 - scripts by A.F. Wells (2025)")
    bc_pm25.attrs["description"] = description
    bc_pm25.attrs["ensemble_number"] = ens_num
    bc_pm25.attrs["scenario"] = scenario
    bc_pm25.attrs["model"] = model
    bc_pm25.attrs["units"] = "µg/m3"
    bc_pm25.to_netcdf(out_path)

print("All processing complete.")